# Problem & Population:
**What is the problem?** People who have food insecurity in the Bay area struggle with budgeting for food or finding cost effective solutions.

**Who is affected?** food insecure people in the Bay area

# Proposed System
**What is the exact failure point?** There is a lack of information on available food insecurity resources and low-cost cooking methods

**How are Lab 2 and Lab 3 relevant?** Using the structures and image recognition in lab 2 and 3, people can either describe their ingredients/budget or upload an image of their available ingredients and get recipe recommendations and other food information

# Importing & Initializing Gemini

free api key at: [aistudio.google.com](https://www.google.com/url?q=https%3A%2F%2Faistudio.google.com)

In [ ]:
!pip install -q google-generativeai
import google.generativeai as genai
from google.colab import userdata
import json
import time
from google.colab import userdata, files
from IPython.display import display
from PIL import Image as PILImage
import time
import os

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini initialized successfully.")

# Project Code

## Set Up Schema

In [ ]:
schema_prompt = """
Extract information from this list of ingredients provided.
Return ONLY valid JSON with exactly these five fields:
{
    "NUTRITIONAL QUALITY": string (Provide a brief summary of the nutrional quality of each ingredient),
    "RECIPE": string (Reccommend a recipe to make out of the ingredients in the image),
    "PRICE": string (State Price of the meal and if there are any cheaper alternatives)
}
No explanation. No markdown. JSON only.
"""

## input prompt and receive response

In [ ]:
choice = input("Would you like to describe your ingredients or upload an image of them? TYPE: 'describe' or 'upload'")

if choice == 'describe':
    resident_message = (
        input("What are your Ingredients?")
    )
    def extract_structured(message):
        m = genai.GenerativeModel(
            model_name="gemini-2.5-flash",
            system_instruction=schema_prompt
        )
        response = m.generate_content(message)
        time.sleep(12)  # stays under free tier rate limit
        raw = response.text.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return json.loads(raw)

    print("--- Resident message ---")
    print(resident_message)
    print("\n--- Structured extraction (run 3 times — format is identical each time) ---")

    for i in range(1, 4):
        print(f"\nRun {i}:")
        result = extract_structured(resident_message)
        print(json.dumps(result, indent=2, ensure_ascii=False))
elif choice == 'upload':
    uploaded = files.upload()

    # Get the filename of the uploaded file.
    image_filename = list(uploaded.keys())[0]

    # Display the uploaded image so you can see what the model will analyze.
    img = PILImage.open(image_filename)
    print(f"Uploaded: {image_filename}")
    print(f"Image size: {img.size[0]}x{img.size[1]} pixels")
    display(img)

    def analyze_image(image_path, question):
          """
          Send an image + a question to Gemini and return the response.
          Returns: (response_text, usage_metadata)
          """
          m = genai.GenerativeModel(model_name="gemini-2.5-flash")
          img = PILImage.open(image_path)
          response = m.generate_content([question, img])
          time.sleep(12)  # stays under free tier rate limit
          return response.text, response.usage_metadata

          print("analyze_image() function defined. Ready to use in Parts 3 and 4.")

    civic_questions = [
    ("INGREDIENTS",  "Describe the ingredients shown in the image. Be specific about what you see."),
    ("NUTRITIONAL QUALITY",   "Provide a brief summary of the nutrional quality of each ingredient"),
    ("RECIPE",  "Reccommend a recipe to make out of the ingredients in the image"),
    ("PRICE",   "State Price of the meal and if there are any cheaper alternatives")
    ]

    civic_results = {"answers": {}, "total_tokens": 0}

    for label, question in civic_questions:
        print(f"--- {label} ---")
        answer, usage = analyze_image(image_filename, question)
        civic_results["answers"][label] = answer
        civic_results["total_tokens"] += usage.total_token_count
        print(answer)
        print()

    print(f"--- Total tokens used: {civic_results['total_tokens']} ---")
    print("Running on Gemini free tier — no cost.")

TypeError: 'NoneType' object is not subscriptable

# Edge Case

Uploaded an image of trash instead of food. System recognized the image as trash and recommended user not to consume and passed by recipe and price outputs. Response was acceptable, but ran through the entire structure and took a long time. Edited code for addressing cases like this more efficiently is below.

In [ ]:
choice = input("Would you like to describe your ingredients or upload an image of them? TYPE: 'describe' or 'upload'")

if choice == 'describe':
    resident_message = (
        input("What are your Ingredients?")
    )
    def extract_structured(message):
        m = genai.GenerativeModel(
            model_name="gemini-2.5-flash",
            system_instruction=schema_prompt
        )
        response = m.generate_content(message)
        time.sleep(12)  # stays under free tier rate limit
        raw = response.text.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        return json.loads(raw)

    print("--- Resident message ---")
    print(resident_message)
    print("\n--- Structured extraction (run 3 times — format is identical each time) ---")

    for i in range(1, 4):
        print(f"\nRun {i}:")
        result = extract_structured(resident_message)
        print(json.dumps(result, indent=2, ensure_ascii=False))
elif choice == 'upload':
    uploaded = files.upload()

    # Get the filename of the uploaded file.
    image_filename = list(uploaded.keys())[0]

    # Display the uploaded image so you can see what the model will analyze.
    img = PILImage.open(image_filename)
    print(f"Uploaded: {image_filename}")
    print(f"Image size: {img.size[0]}x{img.size[1]} pixels")
    display(img)

    def analyze_image(image_path, question):
          """
          Send an image + a question to Gemini and return the response.
          Returns: (response_text, usage_metadata)
          """
          m = genai.GenerativeModel(model_name="gemini-2.5-flash")
          img = PILImage.open(image_path)
          response = m.generate_content([question, img])
          time.sleep(12)  # stays under free tier rate limit
          return response.text, response.usage_metadata

    # First, check if the image contains food ingredients
    food_check_question = "Does the image primarily depict food ingredients suitable for cooking? Answer with 'Yes' or 'No' only."
    food_check_response, _ = analyze_image(image_filename, food_check_question)

    if "no" in food_check_response.lower():
        print("\n--- Non-food image detected ---")
        print("The uploaded image does not appear to contain food ingredients. Please upload an image of food ingredients to get recipe recommendations.")
    else:
        print("\n--- Food ingredients detected. Proceeding with analysis ---")
        civic_questions = [
        ("INGREDIENTS",  "Describe the ingredients shown in the image. Be specific about what you see."),
        ("NUTRITIONAL QUALITY",   "Provide a brief summary of the nutrional quality of each ingredient"),
        ("RECIPE",  "Reccommend a recipe to make out of the ingredients in the image"),
        ("PRICE",   "State Price of the meal and if there are any cheaper alternatives")
        ]

        civic_results = {"answers": {}, "total_tokens": 0}

        for label, question in civic_questions:
            print(f"--- {label} ---")
            answer, usage = analyze_image(image_filename, question)
            civic_results["answers"][label] = answer
            civic_results["total_tokens"] += usage.total_token_count
            print(answer)
            print()

        print(f"--- Total tokens used: {civic_results['total_tokens']} ---")
        print("Running on Gemini free tier — no cost.")
else:
    print("Invalid choice. Please enter 'describe' or 'upload'.")